### Crear embeddings

In [2]:
# -*- coding: utf-8 -*-
import os
import torch
import numpy as np
import pandas as pd
import sys
import json
from types import SimpleNamespace

# Ajusta esto si es necesario para importar tu modelo TuckER
sys.path.append('../..')
from model import TuckER

# ============================================================
# 🔹 CONFIGURACIÓN
# ============================================================

# Carpeta del Grafo (Dataset 2019-2)
data_dir = r"C:\Users\56946\TuckER\data\dataset_20192_fundamentales"

# ⚠️ CARPETA DE SALIDA SOLICITADA
output_dir = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start"
os.makedirs(output_dir, exist_ok=True)

# Archivos con notas (Historia: Solo 2019-1)
base_path = r"C:/Users/56946/TuckER/mis_scripts/dataframes_por_semestre"
csv_20191 = os.path.join(base_path, "df_20191.csv")

# Archivo de Puntajes
path_puntajes = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

# Cursos fundamentales (4 dimensiones)
cursos_1er_semestre = ["MA1101", "MA1001", "FI1000", "BT1211"]

# Dimensiones del modelo (5 para incluir el puntaje)
edim, rdim = 5, 10
kwargs = {"input_dropout": 0.2, "hidden_dropout1": 0.2, "hidden_dropout2": 0.3}

# ============================================================
# 🔹 FUNCIONES AUXILIARES
# ============================================================

def leer_y_normalizar(path):
    df = pd.read_csv(path, sep=";")
    df.columns = df.columns.str.strip().str.upper()
    df["ID"] = df["ID"].astype(str).str.strip().str.upper()
    df["CURSO"] = df["CURSO"].astype(str).str.strip().str.upper()
    return df

def get_vocab_from_data_dir(data_dir):
    entities = set()
    relations = set()
    nombres = ["train.txt", "valid.txt", "test.txt", "train_balanceado.txt"]
    
    for file_name in nombres:
        path = os.path.join(data_dir, file_name)
        if not os.path.exists(path): continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                parts = line.strip().split()
                if len(parts) >= 3:
                    h, r, t = parts[:3]
                    entities.add(h); entities.add(t); relations.add(r)
    
    relations_with_reverse = sorted(list(relations)) + [r + "_reverse" for r in sorted(list(relations))]
    return sorted(list(entities)), sorted(list(relations_with_reverse))

def heads_desde_tripletas(data_dir):
    heads = set()
    nombres = ["train.txt", "valid.txt", "test.txt"]
    for fname in nombres:
        path = os.path.join(data_dir, fname)
        if not os.path.exists(path): continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                parts = line.strip().split()
                if len(parts) >= 1:
                    heads.add(parts[0])
    return heads

# ============================================================
# 🔹 1. CARGAR Y NORMALIZAR NOTAS (Rango -1 a 1)
# ============================================================

print("Cargando notas académicas (2019-1)...")
df_20191 = leer_y_normalizar(csv_20191)

# Identificar generación
ids_gen_2019 = set(df_20191["ID"].unique())
print(f"   Total alumnos en 2019-1: {len(ids_gen_2019)}")

df_20191['NOTA'] = pd.to_numeric(df_20191['NOTA'], errors='coerce')

# Normalización Notas: 1.0 -> -1.0, 7.0 -> 1.0
def escalar_nota(n):
    if pd.isna(n): return -1.0
    return (n - 4.0) / 3.0

df_20191['NOTA'] = df_20191['NOTA'].apply(escalar_nota)

def obtener_notas(df, cursos):
    return (df[df["CURSO"].isin(cursos)]
            .pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
            .reindex(columns=cursos).fillna(-1.0))

notas_20191 = obtener_notas(df_20191, cursos_1er_semestre)

# ============================================================
# 🔹 2. CARGAR Y NORMALIZAR PUNTAJES (Rango -1 a 1)
# ============================================================

print("Cargando y normalizando puntajes...")
if os.path.exists(path_puntajes):
    df_puntajes = pd.read_csv(path_puntajes, sep=";")
    df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
    df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
    df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
    
    # Obtener puntajes solo de la generación 2019 para calcular min/max local
    scores_gen = df_puntajes[df_puntajes["ID"].isin(ids_gen_2019)]["PUNTAJE_PONDERADO"].dropna()
    
    min_score = scores_gen.min() if not scores_gen.empty else 450.0
    max_score = scores_gen.max() if not scores_gen.empty else 850.0
    
    print(f"   Rango Puntaje detectado (Gen 2019): [{min_score}, {max_score}]")
    
    mapa_puntajes = {}
    
    # Función MinMax a [-1, 1]
    def minmax_scale(val, min_v, max_v):
        if max_v == min_v: return 0.0
        return 2 * (val - min_v) / (max_v - min_v) - 1

    for _, row in df_puntajes.iterrows():
        uid = row["ID"]
        score = row["PUNTAJE_PONDERADO"]
        
        if pd.isna(score):
            val_norm = -1.0 # Valor para missing data
        else:
            # Normalización lineal al rango [-1, 1] usando los límites de la generación
            val_norm = minmax_scale(score, min_score, max_score)
            
        mapa_puntajes[uid] = val_norm
else:
    print(f"⚠️ NO SE ENCONTRÓ EL ARCHIVO DE PUNTAJES: {path_puntajes}")
    mapa_puntajes = {}

# ============================================================
# 🔹 3. VOCABULARIO DEL MODELO (2019-2)
# ============================================================

entities, relations = get_vocab_from_data_dir(data_dir)
print(f"✅ Vocabulario Dataset 2019-2: {len(entities)} entidades, {len(relations)} relaciones.")

d = SimpleNamespace()
d.entities = entities
d.relations = relations
d.entity_idxs = {e: i for i, e in enumerate(entities)}
d.relation_idxs = {r: i for i, r in enumerate(relations)}

# ============================================================
# 🔹 4. ASIGNAR EMBEDDINGS (Dim 5)
# ============================================================

modelo = TuckER(d, edim, rdim, **kwargs)
alumnos_tripletas = sorted([h for h in heads_desde_tripletas(data_dir) if h in d.entity_idxs])

contador_inicializados = 0
contador_sin_datos = 0

print(f"Inyectando vectores de dimensión {edim}...")

with torch.no_grad():
    for alumno in alumnos_tripletas:
        # 1. Obtener Notas (Dim 4)
        if alumno in notas_20191.index:
            notas_vec = notas_20191.loc[alumno].values.astype(np.float32)
            contador_inicializados += 1
        else:
            # Si no está en el registro 2019-1, se llena con -1 (desconocido)
            notas_vec = np.full(len(cursos_1er_semestre), -1.0, dtype=np.float32)
            contador_sin_datos += 1

        # 2. Obtener Puntaje (Dim 1)
        puntaje_val = mapa_puntajes.get(alumno, -1.0)
        
        # 3. Concatenar -> Vector final de 5 elementos
        vector_final = np.append(notas_vec, puntaje_val)
        
        # 4. Asignar al tensor
        idx = d.entity_idxs[alumno]
        modelo.E.weight[idx, :len(vector_final)] = torch.tensor(vector_final, dtype=torch.float32)

print(f"\n📊 Resumen:")
print(f"   Alumnos con historia (2019-1) : {contador_inicializados}")
print(f"   Alumnos sin historia (Cold)   : {contador_sin_datos}")

# ============================================================
# 🔹 5. GUARDAR
# ============================================================

embeddings_path = os.path.join(output_dir, "embeddings_inicializados_normalizados_5d.pt")
vocab_path = os.path.join(output_dir, "vocabulario_5d.json")

torch.save(modelo.E.weight.data, embeddings_path)
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump({"entities": d.entities, "relations": d.relations}, f, indent=2, ensure_ascii=False)

print(f"\n💾 Embeddings 5D guardados en: {embeddings_path}")
print(f"💾 Vocabulario guardado en: {vocab_path}")
print("✅ Listo para entrenar TuckER.")

Cargando notas académicas (2019-1)...
   Total alumnos en 2019-1: 2329
Cargando y normalizando puntajes...
   Rango Puntaje detectado (Gen 2019): [650.15, 935.2]
✅ Vocabulario Dataset 2019-2: 798 entidades, 4 relaciones.
Inyectando vectores de dimensión 5...

📊 Resumen:
   Alumnos con historia (2019-1) : 790
   Alumnos sin historia (Cold)   : 0

💾 Embeddings 5D guardados en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\embeddings_inicializados_normalizados_5d.pt
💾 Vocabulario guardado en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\vocabulario_5d.json
✅ Listo para entrenar TuckER.


# entrenar redes neuronales

### balanceada

In [8]:
# -*- coding: utf-8 -*-
import os, sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from types import SimpleNamespace

# ================================================================
# 🔹 CONFIGURACIÓN GENERAL
# ================================================================
DEVICE = torch.device("cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# --- RUTAS DE ENTRADA ---
# Dataset usado para entrenar el TuckER del 2do Semestre
TUCKER_DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_20192_fundamentales"
RESULTS_BASE    = r"C:\Users\56946\TuckER\results"

# ⚠️ PATRÓN DEL TUCKER (Ajusta si el nombre de tu carpeta es distinto)
# Asumo que usaste este prefijo en el .bat anterior para S1->S2
RUN_PREFIX      = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019patience400_balanceado_5d"

# --- RUTA DE SALIDA ---
SAVE_BASE       = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_sem1_balanceados_5d"

# --- DATOS ---
RUTA_DF_INPUT  = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre\df_20191.csv" # Historia S1
RUTA_DF_TARGET = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre\df_20192.csv" # Target S2 (para balanceo)
RUTA_PUNTAJES  = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

# Cursos
CURSOS_PRIMER  = ['MA1101', 'MA1001', 'FI1000', 'BT1211']
CURSOS_SEGUNDO = ['MA1002', 'MA1102', 'FI1100', 'CC1002']
# Para identificar reprobados miramos si fallaron algo en el segundo semestre
CURSOS_EVAL    = CURSOS_PRIMER + CURSOS_SEGUNDO

RDIMS = range(1, 17)

# ================================================================
# 🔹 FUNCIONES AUXILIARES
# ================================================================

def get_vocab_from_data_dir(data_dir):
    entities = set()
    # Buscamos archivos de vocabulario
    nombres = ['train.txt', 'train_balanceado.txt', 'valid.txt', 'test.txt']
    for part in nombres:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    h, r, t = line.strip().split()
                    entities.add(h.strip().upper())
    return sorted(list(entities))

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def cargar_E_weights(path_pt):
    sd = pick_state_dict(torch.load(path_pt, map_location=DEVICE))
    if "E.weight" not in sd: raise KeyError("E.weight no encontrado en checkpoint")
    return sd["E.weight"].detach().cpu()

def limpiar_nota(nota_str, estado):
    if isinstance(estado, str) and "Reprobado" in estado: return 1.0
    try:
        if pd.isna(nota_str) or str(nota_str).strip() == "": return 0.0
        return float(str(nota_str).replace(",", "."))
    except: return 0.0

# ⚠️ VECTOR 5D (Notas S1 + Puntaje)
def preparar_X_historico_5dim(ruta_input, ruta_puntajes, cursos_primer):
    print(f"🔄 Construyendo vectores históricos de dimensión 5...")
    
    # 1. Cargar Notas
    df = pd.read_csv(ruta_input, sep=';')
    df["ID"] = df["ID"].astype(str).str.strip().str.upper()
    todos_alumnos = df["ID"].unique()
    
    # 2. Cargar y Normalizar Puntajes
    df_puntajes = pd.read_csv(ruta_puntajes, sep=';')
    df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
    df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
    df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
    
    # Máximo local para normalizar (Generación 2019)
    scores_gen = df_puntajes[df_puntajes["ID"].isin(todos_alumnos)]["PUNTAJE_PONDERADO"]
    max_score = scores_gen.max()
    min_score = scores_gen.min()
    
    if pd.isna(max_score): max_score = 850.0; min_score = 450.0
    print(f"   Rango Puntaje detectado: [{min_score}, {max_score}]")

    mapa_puntajes = df_puntajes.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
    idx_primer = {c: i for i, c in enumerate(cursos_primer)}
    vectores = {id_al: np.zeros(5, dtype=np.float32) for id_al in todos_alumnos}
    
    # Función MinMax a [-1, 1]
    def minmax_scale(val):
        if max_score == min_score: return 0.0
        return 2 * (val - min_score) / (max_score - min_score) - 1

    # Llenar Notas (0-3)
    for _, row in df.iterrows():
        id_al = row["ID"]
        curso = row["CURSO"]
        if curso in idx_primer:
            # Nota normalizada [-1, 1] -> (Nota-4)/3
            val = (limpiar_nota(row["NOTA"], row["ESTADO_CURSO"]) - 4.0) / 3.0
            vectores[id_al][idx_primer[curso]] = val

    # Llenar Puntaje (4)
    for id_al in vectores:
        score = mapa_puntajes.get(id_al, np.nan)
        if pd.isna(score):
            vectores[id_al][4] = -1.0 # Sin dato
        else:
            vectores[id_al][4] = minmax_scale(score)

    return pd.DataFrame.from_dict(vectores, orient='index')

def obtener_ids_reprobados(ruta_target, cursos_eval):
    df = pd.read_csv(ruta_target, sep=';')
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df = df[df['CURSO'].isin(cursos_eval)]
    reprobados = set()
    for _, row in df.iterrows():
        estado = str(row['ESTADO_CURSO'])
        nota = row['NOTA']
        es_repro = False
        if "Reprobado" in estado: es_repro = True
        elif "Aprobado" not in estado:
            try: 
                if float(str(nota).replace(",", ".")) < 4.0: es_repro = True
            except: pass
        if es_repro: reprobados.add(row['ID'])
    return reprobados

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

# ENTRENAMIENTO BALANCEADO
def entrenar_predictor_balanceado(X_data, Y_data, ids_comunes, ids_reprobados_set, save_path):
    ids_train_val, ids_test = train_test_split(ids_comunes, test_size=0.20, random_state=42)
    ids_train, ids_val      = train_test_split(ids_train_val, test_size=0.15, random_state=42)
    
    id_to_idx = {uid: i for i, uid in enumerate(ids_comunes)}
    idxs_train = [id_to_idx[uid] for uid in ids_train]
    idxs_val   = [id_to_idx[uid] for uid in ids_val]
    idxs_test  = [id_to_idx[uid] for uid in ids_test]
    
    # Balanceo (Oversampling)
    idxs_reprobados_train = [i for i, uid in zip(idxs_train, ids_train) if uid in ids_reprobados_set]
    n_apr = len(idxs_train) - len(idxs_reprobados_train)
    n_rep = len(idxs_reprobados_train)
    
    final_idxs_train = list(idxs_train)
    if n_rep > 0 and n_apr > n_rep:
        factor = n_apr // n_rep
        final_idxs_train += idxs_reprobados_train * factor
        # print(f"   Balanceo: {n_rep} originales -> {n_rep * factor} clonados")
    
    X_train_t = torch.FloatTensor(X_data[final_idxs_train])
    Y_train_t = torch.FloatTensor(Y_data[final_idxs_train])
    X_val_t = torch.FloatTensor(X_data[idxs_val])
    Y_val_t = torch.FloatTensor(Y_data[idxs_val])
    X_test_t = torch.FloatTensor(X_data[idxs_test])
    Y_test_t = torch.FloatTensor(Y_data[idxs_test])
    
    predictor = EmbeddingPredictor(X_train_t.shape[1], Y_train_t.shape[1]).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(predictor.parameters(), lr=1e-3)
    best_loss, patience, counter = float('inf'), 200, 0

    for epoch in range(1000):
        predictor.train()
        optimizer.zero_grad()
        loss = criterion(predictor(X_train_t), Y_train_t)
        loss.backward(); optimizer.step()

        predictor.eval()
        with torch.no_grad():
            vloss = criterion(predictor(X_val_t), Y_val_t)

        if vloss.item() < best_loss - 1e-9:
            best_loss = vloss.item()
            torch.save(predictor.state_dict(), save_path)
            counter = 0
        else:
            counter += 1
            if counter >= patience: break

    predictor.load_state_dict(torch.load(save_path, map_location=DEVICE))
    predictor.eval()
    with torch.no_grad():
        test_mse = criterion(predictor(X_test_t), Y_test_t).item()
    return test_mse

# ================================================================
# 🔹 MAIN
# ================================================================
def main():
    os.makedirs(SAVE_BASE, exist_ok=True)
    
    print("--- Fase 0: Identificando Reprobados S2 (Target) ---")
    ids_reprobados = obtener_ids_reprobados(RUTA_DF_TARGET, CURSOS_EVAL)
    print(f"   -> Alumnos que reprueban algo en S2: {len(ids_reprobados)}")

    print("\n--- Fase 1: Cargando Vocabulario TuckER ---")
    alumnos_tucker = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    
    print("\n--- Fase 2: Preparando Vectores 5D (4 Notas + Puntaje) ---")
    df_vectores = preparar_X_historico_5dim(RUTA_DF_INPUT, RUTA_PUNTAJES, CURSOS_PRIMER)
    
    vocab_completo = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    entity_idxs = {e: i for i, e in enumerate(vocab_completo)}
    
    alumnos_comunes = sorted(set(df_vectores.index).intersection(vocab_completo))
    print(f"   -> Alumnos Comunes (Trainable): {len(alumnos_comunes)}")

    if not alumnos_comunes:
        print("❌ Error: No hay intersección de alumnos.")
        return

    X_data_total = df_vectores.loc[alumnos_comunes].values.astype(np.float32)

    resumen = []
    print("\n=== Bucle por RDIM (Entrenamiento 5D Balanceado S1->S2) ===")
    for rdim in RDIMS:
        tucker_folder = RUN_PREFIX.format(rdim=rdim)
        tucker_pt = os.path.join(RESULTS_BASE, tucker_folder, "best_model.pt")
        save_path = os.path.join(SAVE_BASE, f"best_predictor_dim5_rdim{rdim}_balanceado_5d.pt")

        print(f"\n>> rdim={rdim} | checkpoint: {tucker_pt}")
        if not os.path.exists(tucker_pt):
            print(f"   ⚠️ No existe checkpoint TuckER. Se omite.")
            continue

        try:
            E = cargar_E_weights(tucker_pt)
            idxs_tucker = [entity_idxs[a] for a in alumnos_comunes]
            Y_data_total = E[idxs_tucker].numpy()

            test_mse = entrenar_predictor_balanceado(
                X_data_total, Y_data_total, 
                alumnos_comunes, ids_reprobados, 
                save_path
            )
            
            print(f"   ✅ Guardado: {save_path}")
            print(f"   📏 MSE (test): {test_mse:.6f}")
            resumen.append((rdim, E.shape[1], len(alumnos_comunes), test_mse, save_path))
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            import traceback
            traceback.print_exc()
            continue

    if resumen:
        df_sum = pd.DataFrame(resumen, columns=["rdim","edim","n_alumnos","test_mse","ruta_modelo"])
        out_csv = os.path.join(SAVE_BASE, "resumen_nn_sem1_balanceado_5d.csv")
        df_sum.sort_values("rdim").to_csv(out_csv, index=False)
        print(f"\n=== Resumen guardado: {out_csv} ===")
        print(df_sum.sort_values("rdim").to_string(index=False))

if __name__ == "__main__":
    main()

--- Fase 0: Identificando Reprobados S2 (Target) ---
   -> Alumnos que reprueban algo en S2: 55

--- Fase 1: Cargando Vocabulario TuckER ---

--- Fase 2: Preparando Vectores 5D (4 Notas + Puntaje) ---
🔄 Construyendo vectores históricos de dimensión 5...
   Rango Puntaje detectado: [650.15, 935.2]
   -> Alumnos Comunes (Trainable): 790

=== Bucle por RDIM (Entrenamiento 5D Balanceado S1->S2) ===

>> rdim=1 | checkpoint: C:\Users\56946\TuckER\results\Experimento_warm_start_rdim1_1000epochs_earlystopping_2019patience400_balanceado_5d\best_model.pt
   ✅ Guardado: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_sem1_balanceados_5d\best_predictor_dim5_rdim1_balanceado_5d.pt
   📏 MSE (test): 0.010576

>> rdim=2 | checkpoint: C:\Users\56946\TuckER\results\Experimento_warm_start_rdim2_1000epochs_earlystopping_2019patience400_balanceado_5d\best_model.pt
   ✅ Guardado: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_sem1_balanceados_5d\best_predictor_dim5_rdi

### no balanceada

In [4]:
# -*- coding: utf-8 -*-
import os, sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from types import SimpleNamespace

# =========================
# CONFIGURACIÓN GENERAL
# =========================
DEVICE = torch.device("cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# --- RUTAS ---
# Dataset del grafo (usado para el vocabulario y para cargar TuckER)
TUCKER_DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_20192_fundamentales"
RESULTS_BASE    = r"C:\Users\56946\TuckER\results"

# Prefijo del TuckER 5D (que ya tienes entrenado)
RUN_PREFIX      = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019patience400_balanceado_5d"

# ⚠️ CARPETA DE SALIDA: ESTÁNDAR (SIN BALANCEO DE RED)
SAVE_BASE       = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_5d_standard"

# --- DATOS ---
RUTA_DF_INPUT  = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre\df_20191.csv"
RUTA_PUNTAJES  = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

CURSOS_PRIMER  = ['MA1101', 'MA1001', 'FI1000', 'BT1211']

RDIMS = range(1, 17)

# =========================
# FUNCIONES AUXILIARES
# =========================
def get_vocab_from_data_dir(data_dir):
    entities = set()
    nombres = ['train.txt', 'train_balanceado.txt', 'valid.txt', 'test.txt']
    for part in nombres:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    h, r, t = line.strip().split()
                    entities.add(h.strip().upper())
    return sorted(list(entities))

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def cargar_E_weights(path_pt):
    sd = pick_state_dict(torch.load(path_pt, map_location=DEVICE))
    return sd["E.weight"].detach().cpu()

def limpiar_nota(nota_str, estado):
    if isinstance(estado, str) and "Reprobado" in estado: return 1.0
    try:
        if pd.isna(nota_str) or str(nota_str).strip() == "": return 0.0
        return float(str(nota_str).replace(",", "."))
    except: return 0.0

# VECTOR 5D (Notas + Puntaje)
def preparar_X_historico_5dim(ruta_input, ruta_puntajes, cursos_primer):
    print(f"🔄 Construyendo vectores históricos de dimensión 5 (4 Notas + Puntaje)...")
    
    df = pd.read_csv(ruta_input, sep=';')
    df["ID"] = df["ID"].astype(str).str.strip().str.upper()
    todos_alumnos = df["ID"].unique()
    
    df_puntajes = pd.read_csv(ruta_puntajes, sep=';')
    df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
    df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
    df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
    
    max_score = df_puntajes[df_puntajes["ID"].isin(todos_alumnos)]["PUNTAJE_PONDERADO"].max()
    if pd.isna(max_score): max_score = 850.0
    print(f"   Max Puntaje detectado: {max_score}")

    mapa_puntajes = df_puntajes.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
    idx_primer = {c: i for i, c in enumerate(cursos_primer)}
    vectores = {id_al: np.zeros(5, dtype=np.float32) for id_al in todos_alumnos}
    
    for _, row in df.iterrows():
        id_al = row["ID"]
        curso = row["CURSO"]
        if curso in idx_primer:
            # Normalización estándar para la red (0-1)
            val = limpiar_nota(row["NOTA"], row["ESTADO_CURSO"]) / 7.0
            vectores[id_al][idx_primer[curso]] = val

    for id_al in vectores:
        score = mapa_puntajes.get(id_al, np.nan)
        if pd.isna(score):
            vectores[id_al][4] = -1.0 
        else:
            vectores[id_al][4] = score / max_score

    return pd.DataFrame.from_dict(vectores, orient='index')

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

# ⚠️ ENTRENAMIENTO ESTÁNDAR (SIN BALANCEO)
def entrenar_predictor_estandar(X_data, Y_data, save_path):
    # Split aleatorio simple (sin lógica de reprobados)
    X_train_val, X_test, Y_train_val, Y_test = train_test_split(X_data, Y_data, test_size=0.20, random_state=42)
    X_train, X_val, Y_train, Y_val = train_test_split(X_train_val, Y_train_val, test_size=0.15, random_state=42)
    
    X_train_t = torch.FloatTensor(X_train)
    Y_train_t = torch.FloatTensor(Y_train)
    X_val_t   = torch.FloatTensor(X_val)
    Y_val_t   = torch.FloatTensor(Y_val)
    X_test_t  = torch.FloatTensor(X_test)
    Y_test_t  = torch.FloatTensor(Y_test)
    
    input_dim = X_train_t.shape[1]
    output_dim = Y_train_t.shape[1]
    
    predictor = EmbeddingPredictor(input_dim, output_dim).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(predictor.parameters(), lr=1e-3)
    best_loss, patience, counter = float('inf'), 200, 0

    for epoch in range(1000):
        predictor.train()
        optimizer.zero_grad()
        loss = criterion(predictor(X_train_t), Y_train_t)
        loss.backward(); optimizer.step()

        predictor.eval()
        with torch.no_grad():
            vloss = criterion(predictor(X_val_t), Y_val_t)

        if vloss.item() < best_loss - 1e-9:
            best_loss = vloss.item()
            torch.save(predictor.state_dict(), save_path)
            counter = 0
        else:
            counter += 1
            if counter >= patience: break

    predictor.load_state_dict(torch.load(save_path, map_location=DEVICE))
    predictor.eval()
    with torch.no_grad():
        test_mse = criterion(predictor(X_test_t), Y_test_t).item()
    return test_mse

# ================================================================
# 🔹 MAIN
# ================================================================
def main():
    os.makedirs(SAVE_BASE, exist_ok=True)
    print(f"📂 Carpeta Salida: {SAVE_BASE}")

    print("\n--- Fase 1: Cargando Vocabulario TuckER ---")
    alumnos_tucker = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    
    print("\n--- Fase 2: Preparando Vectores 5D (4 Notas + Puntaje) ---")
    df_vectores = preparar_X_historico_5dim(RUTA_DF_INPUT, RUTA_PUNTAJES, CURSOS_PRIMER)
    
    vocab_completo = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    entity_idxs = {e: i for i, e in enumerate(vocab_completo)}
    
    alumnos_comunes = sorted(set(df_vectores.index).intersection(vocab_completo))
    print(f"   -> Alumnos Comunes (Trainable): {len(alumnos_comunes)}")

    X_data_total = df_vectores.loc[alumnos_comunes].values.astype(np.float32)

    resumen = []
    print("\n=== Bucle por RDIM (Entrenamiento 5D ESTÁNDAR - SIN BALANCEO) ===")
    for rdim in RDIMS:
        tucker_folder = RUN_PREFIX.format(rdim=rdim)
        tucker_pt = os.path.join(RESULTS_BASE, tucker_folder, "best_model.pt")
        save_path = os.path.join(SAVE_BASE, f"best_predictor_dim5_rdim{rdim}_standard_5d.pt")

        print(f"\n>> rdim={rdim} | checkpoint: {tucker_pt}")
        if not os.path.exists(tucker_pt):
            print(f"   ⚠️ No existe checkpoint TuckER. Se omite.")
            continue

        try:
            E = cargar_E_weights(tucker_pt)
            idxs_tucker = [entity_idxs[a] for a in alumnos_comunes]
            Y_data_total = E[idxs_tucker].numpy()

            test_mse = entrenar_predictor_estandar(
                X_data_total, Y_data_total, 
                save_path
            )
            
            print(f"   ✅ Guardado: {save_path}")
            print(f"   📏 MSE (test): {test_mse:.6f}")
            resumen.append((rdim, E.shape[1], len(alumnos_comunes), test_mse, save_path))
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            import traceback
            traceback.print_exc()
            continue

    if resumen:
        df_sum = pd.DataFrame(resumen, columns=["rdim","edim","n_alumnos","test_mse","ruta_modelo"])
        out_csv = os.path.join(SAVE_BASE, "resumen_nn_sem1_standard_5d.csv")
        df_sum.sort_values("rdim").to_csv(out_csv, index=False)
        print(f"\n=== Resumen guardado: {out_csv} ===")
        print(df_sum.sort_values("rdim").to_string(index=False))

if __name__ == "__main__":
    main()

📂 Carpeta Salida: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_5d_standard

--- Fase 1: Cargando Vocabulario TuckER ---

--- Fase 2: Preparando Vectores 5D (4 Notas + Puntaje) ---
🔄 Construyendo vectores históricos de dimensión 5 (4 Notas + Puntaje)...
   Max Puntaje detectado: 935.2
   -> Alumnos Comunes (Trainable): 790

=== Bucle por RDIM (Entrenamiento 5D ESTÁNDAR - SIN BALANCEO) ===

>> rdim=1 | checkpoint: C:\Users\56946\TuckER\results\Experimento_warm_start_rdim1_1000epochs_earlystopping_2019patience400_balanceado_5d\best_model.pt
   ✅ Guardado: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_5d_standard\best_predictor_dim5_rdim1_standard_5d.pt
   📏 MSE (test): 0.012873

>> rdim=2 | checkpoint: C:\Users\56946\TuckER\results\Experimento_warm_start_rdim2_1000epochs_earlystopping_2019patience400_balanceado_5d\best_model.pt
   ✅ Guardado: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_5d_standard\best_predictor_dim5_rdim2_

# Probar modelos: red sin balancear

In [7]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN DE RUTAS (SOLICITADA)
# ============================================================

# Dataset del Segundo Semestre (Modelo Binario)
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_20192_fundamentales"

# ⚠️ RUTA TUCKER (Tal cual la enviaste)
TUCKER_DIR_FMT = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019patience400_balanceado_5d\best_model.pt"

# ⚠️ RUTA PREDICTORES (Tal cual la enviaste)
PREDICTOR_BASE = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_5d_standard"
PRED_FILE_FMT  = r"best_predictor_dim5_rdim{rdim}_standard_5d.pt"

# Directorios Base
BASE_PATH    = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
RESULTS_BASE = r"C:\Users\56946\TuckER\results"

# Datos de Evaluación (2020)
CSV_HISTORIA  = os.path.join(BASE_PATH, "df_20201.csv") # Input S1 (Gen 2020)
CSV_TARGET    = os.path.join(BASE_PATH, "df_20202.csv") # Target S2
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_EVAL     = CURSOS_PRIMER + CURSOS_SEGUNDO

RDIMS = range(1, 17) 
DEVICE = "cpu"

# ============================================================
# 🔹 UTILIDADES Y MODELO
# ============================================================

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        # Input size = 5 (4 Notas + 1 Puntaje)
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_predictor(predictor_path, input_size, out_dim):
    model = EmbeddingPredictor(input_size, out_dim)
    if not os.path.exists(predictor_path):
        raise FileNotFoundError(f"No existe: {predictor_path}")
    state = torch.load(predictor_path, map_location=DEVICE)
    model.load_state_dict(pick_state_dict(state))
    model.to(DEVICE).eval()
    return model

def load_tucker_weights(tucker_path):
    if not os.path.exists(tucker_path):
        raise FileNotFoundError(f"No existe: {tucker_path}")
    raw = torch.load(tucker_path, map_location=DEVICE)
    state = pick_state_dict(raw)
    return state["E.weight"].to(DEVICE), state["R.weight"].to(DEVICE), state["W"].to(DEVICE)

def get_vocab_manual(data_dir):
    entities, relations = set(), set()
    nombres = ['train.txt', 'train_balanceado.txt', 'valid.txt', 'test.txt', 'train_original.txt']
    for part in nombres:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        h, r, t = parts[:3]
                        entities.add(h.strip().upper())
                        entities.add(t.strip().upper())
                        relations.add(r.strip())
    
    entities = sorted(list(entities))
    relations = sorted(list(relations))
    relations_full = sorted(list(set(relations + [r + "_reverse" for r in relations])))
    
    return SimpleNamespace(
        entities=entities, 
        relations=relations_full,
        entity_idxs={e: i for i, e in enumerate(entities)}, 
        relation_idxs={r: i for i, r in enumerate(relations_full)}
    )

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return rel2idx[cands[0]]

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else: return -9999.0 
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ============================================================
# 🔹 FUNCIONES DE NORMALIZACIÓN
# ============================================================

def escalar_nota_norm(n):
    """Normaliza nota 1.0-7.0 al rango [-1.0, 1.0]"""
    if pd.isna(n): return -1.0
    return (n - 4.0) / 3.0

def minmax_scale(val, min_v, max_v):
    """Normaliza puntaje al rango [-1.0, 1.0]"""
    if max_v == min_v: return 0.0
    return 2 * (val - min_v) / (max_v - min_v) - 1

def get_notes_vector_5d_norm(csv_path, puntajes_map, min_score, max_score, alumno_id, cursos_primer):
    try: df = pd.read_csv(csv_path, sep=';')
    except: return torch.zeros((1, 5), dtype=torch.float32)
    
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    subset['NOTA'] = pd.to_numeric(subset['NOTA'], errors='coerce')
    
    vec_notas = np.full(4, -1.0, dtype=np.float32) 
    idx_map = {c: i for i, c in enumerate(cursos_primer)}
    
    for _, row in subset.iterrows():
        c = row["CURSO"]
        if c in idx_map:
            vec_notas[idx_map[c]] = escalar_nota_norm(row["NOTA"])
            
    score_val = puntajes_map.get(alumno_id, np.nan)
    if pd.isna(score_val):
        puntaje_norm = -1.0
    else:
        puntaje_norm = minmax_scale(score_val, min_score, max_score)
        
    final_vec = np.append(vec_notas, puntaje_norm)
    return torch.tensor(final_vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 MAIN
# ============================================================
def main():
    print("\n--- INICIANDO EVALUACIÓN S1->S2 (5D ESTÁNDAR) ---")
    
    vocab = get_vocab_manual(DATA_DIR)
    
    print("Cargando Datos...")
    df_hist = pd.read_csv(CSV_HISTORIA, sep=';') # 2020-1
    df_tgt  = pd.read_csv(CSV_TARGET, sep=';')
    
    for df in [df_hist, df_tgt]:
        df['ID'] = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    ids_gen_2020 = set(df_hist["ID"].unique())
    
    if os.path.exists(RUTA_PUNTAJES):
        df_ptje = pd.read_csv(RUTA_PUNTAJES, sep=';')
        df_ptje.columns = df_ptje.columns.str.strip().str.upper()
        df_ptje["ID"] = df_ptje["ID"].astype(str).str.strip().str.upper()
        df_ptje["PUNTAJE_PONDERADO"] = pd.to_numeric(df_ptje["PUNTAJE_PONDERADO"], errors='coerce')
        
        scores_gen = df_ptje[df_ptje["ID"].isin(ids_gen_2020)]["PUNTAJE_PONDERADO"].dropna()
        min_score = scores_gen.min() if not scores_gen.empty else 450.0
        max_score = scores_gen.max() if not scores_gen.empty else 850.0
        
        print(f"   Rango Puntaje (Gen 2020): [{min_score}, {max_score}]")
        puntajes_map = df_ptje.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
    else:
        print("⚠️ No se encontró archivo de puntajes. Usando rango default.")
        puntajes_map = {}
        min_score, max_score = 450.0, 850.0

    df_fund = df_hist[df_hist["CURSO"].isin(CURSOS_PRIMER)]
    conteo = df_fund.groupby("ID")["CURSO"].nunique()
    alumnos_validos = set(conteo[conteo == 4].index)
    
    df_eval = df_tgt[(df_tgt['ID'].isin(alumnos_validos)) & (df_tgt['CURSO'].isin(CURSOS_EVAL))].copy()
    df_eval = df_eval[df_eval['CURSO'].isin(vocab.entity_idxs.keys())]
    
    def get_real_label(row):
        estado = str(row['ESTADO_CURSO'])
        if "Aprobado" in estado: return 1 
        if "Reprobado" in estado: return 0 
        try: return 1 if float(str(row['NOTA']).replace(",", ".")) >= 4.0 else 0
        except: return 0
        
    df_eval['y_true'] = df_eval.apply(get_real_label, axis=1)
    print(f"Evaluaciones totales: {len(df_eval)}")
    
    for rdim in RDIMS:
        print(f"\n============================== Evaluando rdim={rdim} ==============================")
        
        tucker_path = os.path.join(RESULTS_BASE, TUCKER_DIR_FMT.format(rdim=rdim))
        predictor_path = os.path.join(PREDICTOR_BASE, PRED_FILE_FMT.format(rdim=rdim))

        if not os.path.exists(tucker_path):
            print(f"⏩ Saltando (Falta TuckER): {tucker_path}")
            continue
        if not os.path.exists(predictor_path):
            print(f"⏩ Saltando (Falta Predictor): {predictor_path}")
            continue

        try:
            E, R, W = load_tucker_weights(tucker_path)
            d1 = E.shape[1]

            idx_apr = find_relation(vocab.relations, vocab.relation_idxs, "aprueba")
            idx_rep = find_relation(vocab.relations, vocab.relation_idxs, "reprueba")

            if idx_apr is None or idx_rep is None:
                print("❌ No se encontraron relaciones aprueba/reprueba.")
                continue

            predictor = load_predictor(predictor_path, input_size=5, out_dim=d1)

            ehat_cache = {}
            for aid in df_eval["ID"].unique():
                x = get_notes_vector_5d_norm(CSV_HISTORIA, puntajes_map, min_score, max_score, aid, CURSOS_PRIMER)
                with torch.no_grad(): ehat_cache[aid] = predictor(x).squeeze(0)

            y_true_bin = []
            y_pred_bin = []

            for _, row in df_eval.iterrows():
                aid = row["ID"]
                curso = row["CURSO"]
                
                if curso not in vocab.entity_idxs: continue
                t_idx = vocab.entity_idxs[curso]
                e_hat = ehat_cache[aid]
                
                s_apr = tucker_score_single(e_hat, R, W, E, idx_apr, t_idx)
                s_rep = tucker_score_single(e_hat, R, W, E, idx_rep, t_idx)
                
                p_apr = torch.sigmoid(torch.tensor(s_apr)).item()
                p_rep = torch.sigmoid(torch.tensor(s_rep)).item()
                
                pred_class = 1 if p_apr >= p_rep else 0
                
                y_true_bin.append(row['y_true'])
                y_pred_bin.append(pred_class)

            tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
            total_reales_reprobados = tn + fp
            total_predichos_reprobados = tn + fn
            
            recall = tn / total_reales_reprobados if total_reales_reprobados > 0 else 0
            precision = tn / total_predichos_reprobados if total_predichos_reprobados > 0 else 0
            accuracy = (tp + tn) / len(y_true_bin)
            
            print(f"📊 Resultados 5D Estándar")
            print("-" * 60)
            print(f"Accuracy Global         : {accuracy:.4f}")
            print(f"Total REALES Reprobados : {total_reales_reprobados}")
            print("-" * 60)
            print(f"✅ RECALL (Sensibilidad) : {recall:.4f} ({tn}/{total_reales_reprobados})")
            print(f"🎯 PRECISION             : {precision:.4f}")
            print(f"Matriz: [TN={tn}] [FP={fp}] | [FN={fn}] [TP={tp}]")

        except Exception as e:
            print(f"❌ Error rdim={rdim}: {e}")

if __name__ == "__main__":
    main()


--- INICIANDO EVALUACIÓN S1->S2 (5D ESTÁNDAR) ---
Cargando Datos...
   Rango Puntaje (Gen 2020): [534.85, 889.1]
Evaluaciones totales: 3021

============================== Evaluando rdim=1 ==============================
📊 Resultados 5D Estándar
------------------------------------------------------------
Accuracy Global         : 0.2800
Total REALES Reprobados : 184
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 0.6902 (127/184)
🎯 PRECISION             : 0.0566
Matriz: [TN=127] [FP=57] | [FN=2118] [TP=719]

============================== Evaluando rdim=2 ==============================
📊 Resultados 5D Estándar
------------------------------------------------------------
Accuracy Global         : 0.7196
Total REALES Reprobados : 184
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 0.2446 (45/184)
🎯 PRECISION             : 0.0598
Matriz: [TN=45] [FP=139] | [FN=708] [TP=2129]

============================== Eval

# Probar modelos: red balanceada

In [9]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN DE RUTAS (SOLICITADA)
# ============================================================

# Dataset del Segundo Semestre (Modelo Binario)
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_20192_fundamentales"

# ⚠️ RUTA TUCKER (Tal cual la enviaste)
TUCKER_DIR_FMT = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019patience400_balanceado_5d\best_model.pt"

# ⚠️ RUTA PREDICTORES (Tal cual la enviaste)
PREDICTOR_BASE = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\predictores_sem1_balanceados_5d"
PRED_FILE_FMT  = r"best_predictor_dim5_rdim{rdim}_balanceado_5d.pt"

# Directorios Base
BASE_PATH    = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
RESULTS_BASE = r"C:\Users\56946\TuckER\results"

# Datos de Evaluación (2020)
CSV_HISTORIA  = os.path.join(BASE_PATH, "df_20201.csv") # Input S1 (Gen 2020)
CSV_TARGET    = os.path.join(BASE_PATH, "df_20202.csv") # Target S2
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_EVAL     = CURSOS_PRIMER + CURSOS_SEGUNDO

RDIMS = range(1, 17) 
DEVICE = "cpu"

# ============================================================
# 🔹 UTILIDADES Y MODELO
# ============================================================

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        # Input size = 5 (4 Notas + 1 Puntaje)
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_predictor(predictor_path, input_size, out_dim):
    model = EmbeddingPredictor(input_size, out_dim)
    if not os.path.exists(predictor_path):
        raise FileNotFoundError(f"No existe: {predictor_path}")
    state = torch.load(predictor_path, map_location=DEVICE)
    model.load_state_dict(pick_state_dict(state))
    model.to(DEVICE).eval()
    return model

def load_tucker_weights(tucker_path):
    if not os.path.exists(tucker_path):
        raise FileNotFoundError(f"No existe: {tucker_path}")
    raw = torch.load(tucker_path, map_location=DEVICE)
    state = pick_state_dict(raw)
    return state["E.weight"].to(DEVICE), state["R.weight"].to(DEVICE), state["W"].to(DEVICE)

def get_vocab_manual(data_dir):
    entities, relations = set(), set()
    nombres = ['train.txt', 'train_balanceado.txt', 'valid.txt', 'test.txt', 'train_original.txt']
    for part in nombres:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        h, r, t = parts[:3]
                        entities.add(h.strip().upper())
                        entities.add(t.strip().upper())
                        relations.add(r.strip())
    
    entities = sorted(list(entities))
    relations = sorted(list(relations))
    relations_full = sorted(list(set(relations + [r + "_reverse" for r in relations])))
    
    return SimpleNamespace(
        entities=entities, 
        relations=relations_full,
        entity_idxs={e: i for i, e in enumerate(entities)}, 
        relation_idxs={r: i for i, r in enumerate(relations_full)}
    )

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return rel2idx[cands[0]]

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else: return -9999.0 
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ============================================================
# 🔹 FUNCIONES DE NORMALIZACIÓN
# ============================================================

def escalar_nota_norm(n):
    """Normaliza nota 1.0-7.0 al rango [-1.0, 1.0]"""
    if pd.isna(n): return -1.0
    return (n - 4.0) / 3.0

def minmax_scale(val, min_v, max_v):
    """Normaliza puntaje al rango [-1.0, 1.0]"""
    if max_v == min_v: return 0.0
    return 2 * (val - min_v) / (max_v - min_v) - 1

def get_notes_vector_5d_norm(csv_path, puntajes_map, min_score, max_score, alumno_id, cursos_primer):
    try: df = pd.read_csv(csv_path, sep=';')
    except: return torch.zeros((1, 5), dtype=torch.float32)
    
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    subset['NOTA'] = pd.to_numeric(subset['NOTA'], errors='coerce')
    
    vec_notas = np.full(4, -1.0, dtype=np.float32) 
    idx_map = {c: i for i, c in enumerate(cursos_primer)}
    
    for _, row in subset.iterrows():
        c = row["CURSO"]
        if c in idx_map:
            vec_notas[idx_map[c]] = escalar_nota_norm(row["NOTA"])
            
    score_val = puntajes_map.get(alumno_id, np.nan)
    if pd.isna(score_val):
        puntaje_norm = -1.0
    else:
        puntaje_norm = minmax_scale(score_val, min_score, max_score)
        
    final_vec = np.append(vec_notas, puntaje_norm)
    return torch.tensor(final_vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 MAIN
# ============================================================
def main():
    print("\n--- INICIANDO EVALUACIÓN S1->S2 (5D ESTÁNDAR) ---")
    
    vocab = get_vocab_manual(DATA_DIR)
    
    print("Cargando Datos...")
    df_hist = pd.read_csv(CSV_HISTORIA, sep=';') # 2020-1
    df_tgt  = pd.read_csv(CSV_TARGET, sep=';')
    
    for df in [df_hist, df_tgt]:
        df['ID'] = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    ids_gen_2020 = set(df_hist["ID"].unique())
    
    if os.path.exists(RUTA_PUNTAJES):
        df_ptje = pd.read_csv(RUTA_PUNTAJES, sep=';')
        df_ptje.columns = df_ptje.columns.str.strip().str.upper()
        df_ptje["ID"] = df_ptje["ID"].astype(str).str.strip().str.upper()
        df_ptje["PUNTAJE_PONDERADO"] = pd.to_numeric(df_ptje["PUNTAJE_PONDERADO"], errors='coerce')
        
        scores_gen = df_ptje[df_ptje["ID"].isin(ids_gen_2020)]["PUNTAJE_PONDERADO"].dropna()
        min_score = scores_gen.min() if not scores_gen.empty else 450.0
        max_score = scores_gen.max() if not scores_gen.empty else 850.0
        
        print(f"   Rango Puntaje (Gen 2020): [{min_score}, {max_score}]")
        puntajes_map = df_ptje.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
    else:
        print("⚠️ No se encontró archivo de puntajes. Usando rango default.")
        puntajes_map = {}
        min_score, max_score = 450.0, 850.0

    df_fund = df_hist[df_hist["CURSO"].isin(CURSOS_PRIMER)]
    conteo = df_fund.groupby("ID")["CURSO"].nunique()
    alumnos_validos = set(conteo[conteo == 4].index)
    
    df_eval = df_tgt[(df_tgt['ID'].isin(alumnos_validos)) & (df_tgt['CURSO'].isin(CURSOS_EVAL))].copy()
    df_eval = df_eval[df_eval['CURSO'].isin(vocab.entity_idxs.keys())]
    
    def get_real_label(row):
        estado = str(row['ESTADO_CURSO'])
        if "Aprobado" in estado: return 1 
        if "Reprobado" in estado: return 0 
        try: return 1 if float(str(row['NOTA']).replace(",", ".")) >= 4.0 else 0
        except: return 0
        
    df_eval['y_true'] = df_eval.apply(get_real_label, axis=1)
    print(f"Evaluaciones totales: {len(df_eval)}")
    
    for rdim in RDIMS:
        print(f"\n============================== Evaluando rdim={rdim} ==============================")
        
        tucker_path = os.path.join(RESULTS_BASE, TUCKER_DIR_FMT.format(rdim=rdim))
        predictor_path = os.path.join(PREDICTOR_BASE, PRED_FILE_FMT.format(rdim=rdim))

        if not os.path.exists(tucker_path):
            print(f"⏩ Saltando (Falta TuckER): {tucker_path}")
            continue
        if not os.path.exists(predictor_path):
            print(f"⏩ Saltando (Falta Predictor): {predictor_path}")
            continue

        try:
            E, R, W = load_tucker_weights(tucker_path)
            d1 = E.shape[1]

            idx_apr = find_relation(vocab.relations, vocab.relation_idxs, "aprueba")
            idx_rep = find_relation(vocab.relations, vocab.relation_idxs, "reprueba")

            if idx_apr is None or idx_rep is None:
                print("❌ No se encontraron relaciones aprueba/reprueba.")
                continue

            predictor = load_predictor(predictor_path, input_size=5, out_dim=d1)

            ehat_cache = {}
            for aid in df_eval["ID"].unique():
                x = get_notes_vector_5d_norm(CSV_HISTORIA, puntajes_map, min_score, max_score, aid, CURSOS_PRIMER)
                with torch.no_grad(): ehat_cache[aid] = predictor(x).squeeze(0)

            y_true_bin = []
            y_pred_bin = []

            for _, row in df_eval.iterrows():
                aid = row["ID"]
                curso = row["CURSO"]
                
                if curso not in vocab.entity_idxs: continue
                t_idx = vocab.entity_idxs[curso]
                e_hat = ehat_cache[aid]
                
                s_apr = tucker_score_single(e_hat, R, W, E, idx_apr, t_idx)
                s_rep = tucker_score_single(e_hat, R, W, E, idx_rep, t_idx)
                
                p_apr = torch.sigmoid(torch.tensor(s_apr)).item()
                p_rep = torch.sigmoid(torch.tensor(s_rep)).item()
                
                pred_class = 1 if p_apr >= p_rep else 0
                
                y_true_bin.append(row['y_true'])
                y_pred_bin.append(pred_class)

            tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
            total_reales_reprobados = tn + fp
            total_predichos_reprobados = tn + fn
            
            recall = tn / total_reales_reprobados if total_reales_reprobados > 0 else 0
            precision = tn / total_predichos_reprobados if total_predichos_reprobados > 0 else 0
            accuracy = (tp + tn) / len(y_true_bin)
            
            print(f"📊 Resultados 5D Estándar")
            print("-" * 60)
            print(f"Accuracy Global         : {accuracy:.4f}")
            print(f"Total REALES Reprobados : {total_reales_reprobados}")
            print("-" * 60)
            print(f"✅ RECALL (Sensibilidad) : {recall:.4f} ({tn}/{total_reales_reprobados})")
            print(f"🎯 PRECISION             : {precision:.4f}")
            print(f"Matriz: [TN={tn}] [FP={fp}] | [FN={fn}] [TP={tp}]")

        except Exception as e:
            print(f"❌ Error rdim={rdim}: {e}")

if __name__ == "__main__":
    main()


--- INICIANDO EVALUACIÓN S1->S2 (5D ESTÁNDAR) ---
Cargando Datos...
   Rango Puntaje (Gen 2020): [534.85, 889.1]
Evaluaciones totales: 3021

============================== Evaluando rdim=1 ==============================
📊 Resultados 5D Estándar
------------------------------------------------------------
Accuracy Global         : 0.2800
Total REALES Reprobados : 184
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 0.6902 (127/184)
🎯 PRECISION             : 0.0566
Matriz: [TN=127] [FP=57] | [FN=2118] [TP=719]

============================== Evaluando rdim=2 ==============================
📊 Resultados 5D Estándar
------------------------------------------------------------
Accuracy Global         : 0.7196
Total REALES Reprobados : 184
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 0.2446 (45/184)
🎯 PRECISION             : 0.0598
Matriz: [TN=45] [FP=139] | [FN=708] [TP=2129]

============================== Eval

### script entrenamiento .bat

In [ ]:
:: ================================================================
:: ENTRENAMIENTO MULTIPLE DE MODELOS TUCKER (5D - NOTAS + PUNTAJE)
:: ================================================================

@echo off
ECHO ===============================================================
ECHO            INICIANDO ENTRENAMIENTOS MULTIDIMENSIONALES (5D)
ECHO ===============================================================

:: Configuración general
set DATASET=dataset_20192_fundamentales
:: ⚠️ RUTAS ACTUALIZADAS A TUS NUEVOS ARCHIVOS 5D
set EMB_INIT=notebooks/Experimento_warm_start/embeddings_5d_final/embeddings_inicializados_normalizados_5d.pt
set VOCAB_INIT=notebooks/Experimento_warm_start/embeddings_5d_final/vocabulario_5d.json

set BATCH=128
set LR=0.003
set PATIENCE=400
set EPOCHS=1000 

:: Lista de dimensiones de relación a entrenar
set RDIMS=1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16

:: Bucle principal
for %%R in (%RDIMS%) do (
    ECHO.
    ECHO ===============================================================
    ECHO Entrenando modelo con dimension de relaciones = %%R
    ECHO ===============================================================

    python main_original_warm_start_gemini_2_earlystopping_gemini.py ^
        --dataset %DATASET% ^
        --output_prefix Experimento_warm_start_rdim%%R_1000epochs_earlystopping_2019patience%PATIENCE%_balanceado_5d ^
        --edim 5 ^
        --rdim %%R ^
        --num_iterations %EPOCHS% ^
        --batch_size %BATCH% ^
        --lr %LR% ^
        --init_embeddings %EMB_INIT% ^
        --init_vocab %VOCAB_INIT% ^
        --patience %PATIENCE%

    ECHO ---------------------------------------------------------------
    ECHO Modelo con rdim=%%R completado.
    ECHO ---------------------------------------------------------------
)

ECHO ===============================================================
ECHO TODOS LOS ENTRENAMIENTOS FINALIZADOS.
ECHO ===============================================================

pause